# Phase 3 — CGRS: Curvature-Guided Rank Selection for ViTs
**USC MS Computer Science (AI) — Spring 2026**

Run this notebook on **Google Colab with a T4/A100 GPU** for best performance.

> **Setup:** Upload `phase1_results/config.json`, `phase1_results/results.json`, and `phase2_results/all_results_complete.json` to your Google Drive before running.

In [ ]:
# ----------------------------------------------------------------
# CELL 1 — Mount Google Drive
# All outputs are saved to Drive so nothing is lost if Colab
# disconnects. Upload these files to your Drive first:
#   MyDrive/CGRS/phase1_results/config.json
#   MyDrive/CGRS/phase1_results/results.json
#   MyDrive/CGRS/phase2_results/all_results_complete.json
# ----------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE  = '/content/drive/MyDrive/CSML Project/Phase1_2_Final/'   # ← change if your folder is different
PHASE1_DIR  = f'{DRIVE_BASE}/phase1_results'
PHASE2_DIR  = f'{DRIVE_BASE}/phase2_results'
PHASE3_DIR  = f'{DRIVE_BASE}/phase3_results'

os.makedirs(PHASE1_DIR, exist_ok=True)
os.makedirs(PHASE2_DIR, exist_ok=True)
os.makedirs(PHASE3_DIR, exist_ok=True)

print(f'Drive mounted.')
print(f'PHASE1_DIR : {PHASE1_DIR}')
print(f'PHASE2_DIR : {PHASE2_DIR}')
print(f'PHASE3_DIR : {PHASE3_DIR}')
print()
# Quick sanity-check — warn if required files are missing
for fpath in [f'{PHASE1_DIR}/config.json', f'{PHASE1_DIR}/results.json']:
    status = '✓ found' if os.path.exists(fpath) else '✗ MISSING — please upload'
    print(f'  {fpath}  →  {status}')
p2path = f'{PHASE2_DIR}/all_results_complete.json'
status = '✓ found' if os.path.exists(p2path) else '⚠ missing (will use default tau values)'
print(f'  {p2path}  →  {status}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.
PHASE1_DIR : /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase1_results
PHASE2_DIR : /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase2_results
PHASE3_DIR : /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase3_results

  /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase1_results/config.json  →  ✓ found
  /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase1_results/results.json  →  ✓ found
  /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase2_results/all_results_complete.json  →  ✓ found


In [ ]:
# ----------------------------------------------------------------
# CELL 2 — Install / upgrade packages (Colab-compatible)
# ----------------------------------------------------------------
!pip install -q --upgrade torch torchvision
!pip install -q --upgrade transformers
!pip install -q --upgrade torchao
!pip install -q --upgrade peft
!pip install -q scipy numpy

# Verify torchao version is compatible
import importlib, pkg_resources
try:
    v = pkg_resources.get_distribution("torchao").version
    print(f"torchao version: {v}")
except Exception:
    print("torchao not found — installing now")
    import subprocess
    subprocess.run(["pip", "install", "-q", "torchao>=0.16.0"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9

/tmp/ipykernel_5107/2603087994.py:11: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import importlib, pkg_resources


In [ ]:
# ----------------------------------------------------------------
# CELL 3 — Imports and reproducibility
# ----------------------------------------------------------------
import os, json, random
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from transformers import ViTForImageClassification, ViTImageProcessor
from peft import LoraConfig, get_peft_model
from scipy.stats import pearsonr, spearmanr

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be very slow on CPU.')
    print('Go to Runtime → Change runtime type → GPU (T4)')


Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


In [ ]:
# ----------------------------------------------------------------
# CELL 4 — Config + load Phase 1/2 results + tau calibration
# PHASE1_DIR / PHASE2_DIR / PHASE3_DIR are set in Cell 1.
# ----------------------------------------------------------------

# Load Phase 1 config and results
with open(f'{PHASE1_DIR}/config.json') as f:
    CONFIG = json.load(f)
with open(f'{PHASE1_DIR}/results.json') as f:
    phase1_results = json.load(f)

LORA_RANKS = [3, 5, 6, 10, 12, 16, 30, 52, 64, 80, 100, 128, 150, 200]
CONFIG['lora_ranks'] = LORA_RANKS

# ---- Load Phase 2 curvature results (for tau calibration) ----
p2_complete_path = f'{PHASE2_DIR}/all_results_complete.json'
phase2_available = os.path.exists(p2_complete_path)

if phase2_available:
    with open(p2_complete_path) as f:
        all_results_p2 = json.load(f)
    curvature_data = all_results_p2.get('curvature', {})
    print('Phase 2 results loaded — computing tau from actual lambda_max values.')
    lmax_by_rank = {}
    for r in LORA_RANKS:
        key = f'r={r}'
        if key in curvature_data:
            lmax_by_rank[r] = curvature_data[key]['lambda_max']
    if lmax_by_rank:
        lmax_values = list(lmax_by_rank.values())
        print(f'  lambda_max range: {min(lmax_values):.6f} — {max(lmax_values):.6f}')
        sorted_lmax = sorted(lmax_values)
        n = len(sorted_lmax)
        tau_high = sorted_lmax[n // 4]
        tau_mid  = sorted_lmax[n // 2]
        tau_low  = sorted_lmax[3 * n // 4]
        TAU_VALUES = [tau_high, tau_mid, tau_low]
        print(f'  Calibrated tau: high={tau_high:.6f}, mid={tau_mid:.6f}, low={tau_low:.6f}')
    else:
        print('  Warning: curvature_data empty — using default tau values.')
        TAU_VALUES = [0.01, 0.05, 0.10]
        lmax_by_rank = {}
else:
    print('Phase 2 results not found — using default tau values.')
    TAU_VALUES = [0.01, 0.05, 0.10]
    lmax_by_rank = {}
    curvature_data = {}

# ---- CGRS hyperparameters ----
CONFIG['cgrs_check_every']   = 200
CONFIG['cgrs_cooldown']      = 100
CONFIG['cgrs_probe_batches'] = 8
CONFIG['cgrs_r_init']        = 16
CONFIG['cgrs_r_min']         = 3
CONFIG['cgrs_r_max']         = 64
CONFIG['cgrs_rank_list']     = [r for r in LORA_RANKS if r <= CONFIG['cgrs_r_max']]
CONFIG['cgrs_tau_values']    = TAU_VALUES

with open(f'{PHASE3_DIR}/config_phase3.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f'\nCGRS config:')
print(f"  r_init       : {CONFIG['cgrs_r_init']}")
print(f"  r_min / r_max: {CONFIG['cgrs_r_min']} / {CONFIG['cgrs_r_max']}")
print(f"  rank_list    : {CONFIG['cgrs_rank_list']}")
print(f"  check_every  : {CONFIG['cgrs_check_every']} steps")
print(f"  cooldown     : {CONFIG['cgrs_cooldown']} steps")
print(f"  probe_batches: {CONFIG['cgrs_probe_batches']} (={CONFIG['cgrs_probe_batches']*4} samples)")
print(f"  tau_values   : {[f'{t:.4f}' for t in TAU_VALUES]}")
print(f'\nPhase 1 baseline results:')
for k, v in phase1_results.items():
    p = v.get('trainable_params', 'n/a')
    print(f"  {k:<22} -> Test Acc: {v['test_acc']:.2f}%  Params: {p}")


Phase 2 results loaded — computing tau from actual lambda_max values.
  lambda_max range: 0.000038 — 0.226160
  Calibrated tau: high=0.000205, mid=0.004680, low=0.051826

CGRS config:
  r_init       : 16
  r_min / r_max: 3 / 64
  rank_list    : [3, 5, 6, 10, 12, 16, 30, 52, 64]
  check_every  : 200 steps
  cooldown     : 100 steps
  probe_batches: 8 (=32 samples)
  tau_values   : ['0.0002', '0.0047', '0.0518']

Phase 1 baseline results:
  Full fine-tune         -> Test Acc: 92.87%  Params: 86567396
  Frozen backbone        -> Test Acc: 86.46%  Params: 76900
  LoRA r=3               -> Test Acc: 84.77%  Params: 110592
  LoRA r=5               -> Test Acc: 87.36%  Params: 184320
  LoRA r=6               -> Test Acc: 87.27%  Params: 221184
  LoRA r=10              -> Test Acc: 89.02%  Params: 368640
  LoRA r=12              -> Test Acc: 89.31%  Params: 442368
  LoRA r=16              -> Test Acc: 89.51%  Params: 589824
  LoRA r=30              -> Test Acc: 90.30%  Params: 1105920
  LoRA r

In [ ]:
# ----------------------------------------------------------------
# CELL 5 — Dataset and DataLoaders
# CIFAR-100 is downloaded automatically — no manual upload needed.
# ----------------------------------------------------------------

# Data is cached locally in Colab (/content/data) for the session
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

try:
    processor = ViTImageProcessor.from_pretrained(CONFIG['model_name'])
    mean = processor.image_mean
    std  = processor.image_std
    print(f"Processor loaded from HuggingFace: {CONFIG['model_name']}")
except Exception:
    mean = [0.5, 0.5, 0.5]
    std  = [0.5, 0.5, 0.5]
    print('No HuggingFace access — using hardcoded ViT-Base normalization.')

print(f'  Normalization mean: {mean}')
print(f'  Normalization std : {std}')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

print('Downloading / loading CIFAR-100...')
full_train   = datasets.CIFAR100(root=DATA_DIR, train=True,  download=True, transform=transform)
test_dataset = datasets.CIFAR100(root=DATA_DIR, train=False, download=True, transform=transform)

val_size = len(full_train) - CONFIG['train_size']
train_dataset, val_dataset = random_split(
    full_train,
    [CONFIG['train_size'], val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# Use more workers on Colab GPU instance
NUM_WORKERS = 2 if device.type == 'cuda' else 0

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG['batch_size'], shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG['batch_size'], shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))
curvature_probe_loader = DataLoader(train_dataset, batch_size=4, shuffle=False,
                                    num_workers=0, pin_memory=False)

print(f'  Train : {len(train_dataset):,}')
print(f'  Val   : {len(val_dataset):,}')
print(f'  Test  : {len(test_dataset):,}')
print(f'  Probe loader: batch_size=4 | {len(curvature_probe_loader)} batches total')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Processor loaded from HuggingFace: google/vit-base-patch16-224
  Normalization mean: (0.5, 0.5, 0.5)
  Normalization std : (0.5, 0.5, 0.5)


100%|██████████| 169M/169M [00:04<00:00, 41.7MB/s]


  Train : 45,000
  Val   : 5,000
  Test  : 10,000
  Probe loader: batch_size=4 | 11250 batches total


In [ ]:
# ----------------------------------------------------------------
# CELL 6 — Model builders and evaluation
# ----------------------------------------------------------------
def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total


def build_lora_model_silent(r):
    base = ViTForImageClassification.from_pretrained(
        CONFIG['model_name'],
        num_labels=CONFIG['num_classes'],
        ignore_mismatched_sizes=True,
    )
    cfg = LoraConfig(
        r              = r,
        lora_alpha     = CONFIG['lora_alpha'],
        lora_dropout   = CONFIG['lora_dropout'],
        target_modules = CONFIG['target_modules'],
        bias           = 'none',
    )
    return get_peft_model(base, cfg).to(device)


def build_lora_model_verbose(r):
    model = build_lora_model_silent(r)
    t, total = count_parameters(model)
    print(f'  LoRA r={r:<4} | Trainable: {t:>10,} / {total:,} ({100*t/total:.3f}%)')
    return model, t


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    loss_sum, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(images)
        loss_sum += criterion(outputs.logits, labels).item()
        correct  += (outputs.logits.argmax(dim=1) == labels).sum().item()
        total    += labels.size(0)
    return loss_sum / len(loader), 100.0 * correct / total


print('Model builders and evaluate() ready.')


Model builders and evaluate() ready.


In [ ]:
# ----------------------------------------------------------------
# CELL 7 — Rank transition and Fisher probe utilities
# ----------------------------------------------------------------
_criterion_probe = nn.CrossEntropyLoss()


def transition_lora_rank(old_model, old_r, new_r):
    old_params = {name: param.data.clone().cpu()
                  for name, param in old_model.named_parameters()}

    new_model = build_lora_model_silent(new_r)
    new_model.cpu()

    new_param_dict = dict(new_model.named_parameters())
    for name, old_data in old_params.items():
        if 'lora_A' not in name and 'lora_B' not in name:
            if name in new_param_dict:
                new_param_dict[name].data.copy_(old_data)

    lora_bases = {}
    for name, data in old_params.items():
        if 'lora_A' in name:
            base = name[:name.index('lora_A')]
            lora_bases.setdefault(base, {})
            lora_bases[base]['A']      = data
            lora_bases[base]['name_A'] = name
        elif 'lora_B' in name:
            base = name[:name.index('lora_B')]
            lora_bases.setdefault(base, {})
            lora_bases[base]['B']      = data
            lora_bases[base]['name_B'] = name

    for base, pair in lora_bases.items():
        if 'A' not in pair or 'B' not in pair:
            continue
        A = pair['A'].float()
        B = pair['B'].float()

        if new_r > old_r:
            new_A = torch.zeros(new_r, A.shape[1])
            new_A[:old_r, :] = A
            new_B = torch.zeros(B.shape[0], new_r)
            new_B[:, :old_r] = B
        else:
            W = B @ A
            try:
                U, S, Vh = torch.linalg.svd(W, full_matrices=False)
                sqrt_S = torch.sqrt(S[:new_r].clamp(min=0.0))
                new_B  = U[:, :new_r] * sqrt_S
                new_A  = Vh[:new_r, :] * sqrt_S.unsqueeze(1)
            except RuntimeError:
                new_A = A[:new_r, :]
                new_B = B[:, :new_r]

        for name, param in new_model.named_parameters():
            if name.startswith(base) and 'lora_A' in name:
                param.data.copy_(new_A.to(param.dtype))
            elif name.startswith(base) and 'lora_B' in name:
                param.data.copy_(new_B.to(param.dtype))

    return new_model.to(device)


def probe_fisher(model, probe_loader, n_batches):
    was_training = model.training
    model.eval()
    fisher_diag  = None
    batches_done = 0

    for images, labels in probe_loader:
        if batches_done >= n_batches:
            break
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        model.zero_grad()
        outputs = model(images)
        loss    = _criterion_probe(outputs.logits, labels)
        loss.backward()
        with torch.no_grad():
            grads = torch.cat([
                p.grad.detach().pow(2).flatten()
                for p in model.parameters()
                if p.requires_grad and p.grad is not None
            ])
        fisher_diag = grads if fisher_diag is None else fisher_diag + grads
        batches_done += 1

    model.zero_grad()
    if was_training:
        model.train()

    if fisher_diag is None or batches_done == 0:
        return float('inf')

    fisher_diag /= batches_done
    return float(fisher_diag.max().item())


print('transition_lora_rank() and probe_fisher() ready.')
print(f"  probe_fisher: {CONFIG['cgrs_probe_batches']} batches x 4 = {CONFIG['cgrs_probe_batches']*4} samples per check")


transition_lora_rank() and probe_fisher() ready.
  probe_fisher: 8 batches x 4 = 32 samples per check


In [ ]:
# ----------------------------------------------------------------
# CELL 8 — CGRS training loop (fully resume-safe)
# Checkpoints are saved to Google Drive after every rank change
# and every epoch — safe to disconnect and resume anytime.
# ----------------------------------------------------------------
CKPT_BASE = f'{PHASE3_DIR}/checkpoints'
os.makedirs(CKPT_BASE, exist_ok=True)


def _ckpt_dir(label):
    return f'{CKPT_BASE}/{label}'


def _save(label, payload, model, optimizer, scheduler):
    d = _ckpt_dir(label)
    os.makedirs(d, exist_ok=True)
    torch.save(model.state_dict(),     f'{d}/model.pt')
    torch.save(optimizer.state_dict(), f'{d}/optimizer.pt')
    torch.save(scheduler.state_dict(), f'{d}/scheduler.pt')
    payload_json = {**payload,
                    'rank_step_count': {str(k): v
                                        for k, v in payload['rank_step_count'].items()}}
    with open(f'{d}/state.json', 'w') as f:
        json.dump(payload_json, f, indent=2)


def _load(label):
    d = _ckpt_dir(label)
    if not os.path.exists(f'{d}/state.json'):
        return None
    with open(f'{d}/state.json') as f:
        state = json.load(f)
    state['rank_step_count'] = defaultdict(
        int, {int(k): v for k, v in state['rank_step_count'].items()}
    )
    return state, d


def run_cgrs(tau, label):
    RANK_LIST   = CONFIG['cgrs_rank_list']
    R_MIN       = CONFIG['cgrs_r_min']
    R_MAX       = CONFIG['cgrs_r_max']
    K           = CONFIG['cgrs_check_every']
    COOLDOWN    = CONFIG['cgrs_cooldown']
    N_PROBE     = CONFIG['cgrs_probe_batches']
    r_init      = CONFIG['cgrs_r_init']
    EPOCHS      = CONFIG['epochs']
    criterion   = nn.CrossEntropyLoss()
    total_steps = EPOCHS * len(train_loader)

    print(f"\n{'='*65}")
    print(f'  {label} | tau={tau:.4f} | r_init={r_init}')
    print(f'  check_every={K} | cooldown={COOLDOWN} | probe={N_PROBE} batches')
    print(f"{'='*65}")

    resume = _load(label)

    if resume is not None:
        state, ckpt_dir   = resume
        start_epoch       = state['start_epoch']
        start_batch       = state['start_batch']
        current_r         = state['current_r']
        global_step       = state['global_step']
        steps_since_chg   = state['steps_since_chg']
        rank_trajectory   = state['rank_trajectory']
        lambda_trajectory = state['lambda_trajectory']
        rank_changes      = state['rank_changes']
        rank_step_count   = state['rank_step_count']
        resume_loss       = state.get('running_loss', 0.0)
        resume_correct    = state.get('correct', 0)
        resume_total      = state.get('total', 0)
        print(f'  [RESUME] epoch={start_epoch} batch={start_batch} r={current_r} step={global_step}')
        model, _ = build_lora_model_verbose(current_r)
        model.load_state_dict(torch.load(f'{ckpt_dir}/model.pt', map_location=device))
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=CONFIG['lr_lora'], weight_decay=CONFIG['weight_decay']
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
        optimizer.load_state_dict(torch.load(f'{ckpt_dir}/optimizer.pt', map_location=device))
        scheduler.load_state_dict(torch.load(f'{ckpt_dir}/scheduler.pt'))
    else:
        start_epoch       = 1
        start_batch       = 0
        current_r         = r_init
        global_step       = 0
        steps_since_chg   = COOLDOWN
        rank_trajectory   = []
        lambda_trajectory = []
        rank_changes      = []
        rank_step_count   = defaultdict(int)
        resume_loss       = 0.0
        resume_correct    = 0
        resume_total      = 0
        print(f'  [FRESH] Starting from epoch=1, r={r_init}')
        torch.cuda.empty_cache()
        model, _ = build_lora_model_verbose(r_init)
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=CONFIG['lr_lora'], weight_decay=CONFIG['weight_decay']
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        if epoch == start_epoch and start_batch > 0:
            running_loss = resume_loss
            correct      = resume_correct
            total        = resume_total
        else:
            running_loss, correct, total = 0.0, 0, 0

        for batch_idx, (images, labels_b) in enumerate(train_loader):
            if epoch == start_epoch and batch_idx < start_batch:
                continue

            images   = images.to(device, non_blocking=True)
            labels_b = labels_b.to(device, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs.logits, labels_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()), max_norm=1.0
            )
            optimizer.step()
            scheduler.step()

            running_loss    += loss.item()
            correct         += (outputs.logits.argmax(1) == labels_b).sum().item()
            total           += labels_b.size(0)
            rank_step_count[current_r] += 1
            global_step     += 1
            steps_since_chg += 1

            # ---- RANK CHECK ----
            if steps_since_chg >= K:
                lmax = probe_fisher(model, curvature_probe_loader, N_PROBE)
                rank_trajectory.append((global_step, current_r))
                lambda_trajectory.append((global_step, lmax))

                rank_idx = (RANK_LIST.index(current_r) if current_r in RANK_LIST
                            else min(range(len(RANK_LIST)),
                                     key=lambda i: abs(RANK_LIST[i] - current_r)))
                new_r  = current_r
                reason = 'stable'

                if lmax > tau and rank_idx < len(RANK_LIST) - 1:
                    candidate = RANK_LIST[rank_idx + 1]
                    if candidate <= R_MAX:
                        new_r  = candidate
                        reason = f'sharp (lmax={lmax:.5f} > tau={tau:.5f})'

                elif lmax < tau and rank_idx > 0:
                    candidate = RANK_LIST[rank_idx - 1]
                    if candidate >= R_MIN:
                        new_r  = candidate
                        reason = f'flat (lmax={lmax:.5f} < tau={tau:.5f})'

                if new_r != current_r:
                    direction = 'UP' if new_r > current_r else 'DOWN'
                    print(f'  [Step {global_step:>5}] Rank {direction}: {current_r} → {new_r} | {reason}')
                    rank_changes.append({
                        'step'  : global_step,
                        'old_r' : current_r,
                        'new_r' : new_r,
                        'lambda': lmax,
                        'reason': reason,
                    })
                    current_lr = float(scheduler.get_last_lr()[0])
                    model      = transition_lora_rank(model, current_r, new_r)
                    current_r  = new_r
                    remaining  = max(total_steps - global_step, 1)
                    optimizer  = optim.AdamW(
                        filter(lambda p: p.requires_grad, model.parameters()),
                        lr=current_lr, weight_decay=CONFIG['weight_decay']
                    )
                    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=remaining)
                    torch.cuda.empty_cache()

                    _save(label, {
                        'start_epoch'     : epoch,
                        'start_batch'     : batch_idx + 1,
                        'current_r'       : current_r,
                        'global_step'     : global_step,
                        'steps_since_chg' : 0,
                        'rank_trajectory' : rank_trajectory,
                        'lambda_trajectory': lambda_trajectory,
                        'rank_changes'    : rank_changes,
                        'rank_step_count' : rank_step_count,
                        'running_loss'    : running_loss,
                        'correct'         : correct,
                        'total'           : total,
                    }, model, optimizer, scheduler)
                    print(f'  [SAVED to Drive] After rank change at step {global_step}')

                steps_since_chg = 0

        val_loss, val_acc = evaluate(model, val_loader)
        train_acc = 100.0 * correct / total
        print(f'\n  {label} Epoch {epoch}/{EPOCHS} r={current_r} | '
              f'Train Loss: {running_loss/len(train_loader):.4f} '
              f'Train Acc: {train_acc:.2f}% | '
              f'Val Loss: {val_loss:.4f} Val Acc: {val_acc:.2f}%')

        _save(label, {
            'start_epoch'     : epoch + 1,
            'start_batch'     : 0,
            'current_r'       : current_r,
            'global_step'     : global_step,
            'steps_since_chg' : steps_since_chg,
            'rank_trajectory' : rank_trajectory,
            'lambda_trajectory': lambda_trajectory,
            'rank_changes'    : rank_changes,
            'rank_step_count' : rank_step_count,
            'running_loss'    : 0.0,
            'correct'         : 0,
            'total'           : 0,
        }, model, optimizer, scheduler)
        print(f'  [SAVED to Drive] Epoch {epoch} checkpoint.')
        start_batch = 0

    test_loss, test_acc = evaluate(model, test_loader)
    trainable_params    = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_steps_done    = sum(rank_step_count.values())
    avg_rank            = sum(r * cnt for r, cnt in rank_step_count.items()) / max(total_steps_done, 1)

    print(f'\n  {label} DONE — Test Acc: {test_acc:.2f}% | Avg rank: {avg_rank:.2f} | Changes: {len(rank_changes)}')

    del model
    torch.cuda.empty_cache()

    return {
        'test_acc'         : test_acc,
        'test_loss'        : test_loss,
        'tau'              : tau,
        'r_init'           : r_init,
        'final_rank'       : current_r,
        'avg_rank'         : avg_rank,
        'trainable_params' : trainable_params,
        'n_rank_changes'   : len(rank_changes),
        'rank_changes'     : rank_changes,
        'rank_step_count'  : dict(rank_step_count),
        'rank_trajectory'  : rank_trajectory,
        'lambda_trajectory': lambda_trajectory,
    }


print('run_cgrs() ready — checkpoints saved to Google Drive after every rank change and epoch.')


run_cgrs() ready — checkpoints saved to Google Drive after every rank change and epoch.


In [ ]:
# ----------------------------------------------------------------
# CELL 9 — Load Phase 1 fixed-rank baselines
# ----------------------------------------------------------------
p3_json = f'{PHASE3_DIR}/phase3_results.json'
if os.path.exists(p3_json):
    with open(p3_json) as f:
        phase3_results = json.load(f)
    print(f'Loaded existing phase3_results.json ({len(phase3_results)} entries):')
    for k, v in phase3_results.items():
        if 'test_acc' in v:
            print(f'  {k}: {v["test_acc"]:.2f}%')
else:
    phase3_results = {}
    print('Starting fresh phase3_results dict.')

BASELINE_RANKS = [6, 10, 16, 64]   # ← added 64

print('\n' + '='*70)
print('  FIXED-RANK BASELINES from Phase 1')
print('='*70)

for r in BASELINE_RANKS:
    key_p1 = f'LoRA r={r}'
    if key_p1 in phase1_results:
        v = phase1_results[key_p1]
        phase3_results[f'Fixed r={r}'] = {
            'test_acc'        : v['test_acc'],
            'test_loss'       : v['test_loss'],
            'trainable_params': v.get('trainable_params', 'n/a'),
            'source'          : 'Phase 1',
            'avg_rank'        : float(r),
        }
        print(f"  Fixed r={r:<5} - Test Acc: {v['test_acc']:.2f}%  Test Loss: {v['test_loss']:.4f}")
    else:
        print(f'  Fixed r={r} NOT FOUND in phase1_results')

for key in ['Full fine-tune', 'Frozen backbone']:
    if key in phase1_results:
        v = phase1_results[key]
        phase3_results[key] = {
            'test_acc' : v['test_acc'],
            'test_loss': v['test_loss'],
            'source'   : 'Phase 1',
        }
        print(f'  {key:<22} - Test Acc: {v["test_acc"]:.2f}%')

with open(p3_json, 'w') as f:
    json.dump(phase3_results, f, indent=2)
print(f'\nBaselines loaded and saved to Drive.')

Loaded existing phase3_results.json (9 entries):
  Fixed r=6: 87.27%
  Fixed r=10: 89.02%
  Fixed r=16: 89.51%
  Full fine-tune: 92.87%
  Frozen backbone: 86.46%
  CGRS_tau0.0002: 86.24%
  CGRS_tau0.0047: 90.02%
  CGRS_tau0.0518: 87.96%
  CGRS_tau0.0500: 80.95%

  FIXED-RANK BASELINES from Phase 1
  Fixed r=6     - Test Acc: 87.27%  Test Loss: 0.6846
  Fixed r=10    - Test Acc: 89.02%  Test Loss: 0.5439
  Fixed r=16    - Test Acc: 89.51%  Test Loss: 0.4606
  Fixed r=64    - Test Acc: 90.48%  Test Loss: 0.3840
  Full fine-tune         - Test Acc: 92.87%
  Frozen backbone        - Test Acc: 86.46%

Baselines loaded and saved to Drive.


In [ ]:
# ----------------------------------------------------------------
# CELL 10 — CGRS Run 1: tau_high
# Re-running this cell resumes from the last saved checkpoint.
# ----------------------------------------------------------------
tau1   = TAU_VALUES[0]
label1 = f'CGRS_tau{tau1:.4f}'

if label1 in phase3_results and 'test_acc' in phase3_results[label1]:
    print(f'Already complete: {label1}  acc={phase3_results[label1]["test_acc"]:.2f}%')
else:
    result1 = run_cgrs(tau=tau1, label=label1)
    phase3_results[label1] = result1
    with open(f'{PHASE3_DIR}/phase3_results.json', 'w') as f:
        json.dump(phase3_results, f, indent=2)
    with open(f'{PHASE3_DIR}/trajectory_{label1}.json', 'w') as f:
        json.dump({
            'rank_trajectory'  : result1['rank_trajectory'],
            'lambda_trajectory': result1['lambda_trajectory'],
            'rank_changes'     : result1['rank_changes'],
            'rank_step_count'  : result1['rank_step_count'],
        }, f, indent=2)
    print(f'Saved phase3_results.json to Drive.')



  CGRS_tau0.0002 | tau=0.0002 | r_init=16
  check_every=200 | cooldown=100 | probe=8 batches
  [FRESH] Starting from epoch=1, r=16


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  LoRA r=16   | Trainable:    589,824 / 86,465,380 (0.682%)
  [Step   100] Rank UP: 16 → 30 | sharp (lmax=0.00111 > tau=0.00020)


...

  [SAVED to Drive] After rank change at step 300
  [Step   500] Rank UP: 52 → 64 | sharp (lmax=0.00170 > tau=0.00020)


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  [SAVED to Drive] After rank change at step 500

  CGRS_tau0.0002 Epoch 1/3 r=64 | Train Loss: 2.2620 Train Acc: 58.84% | Val Loss: 1.2208 Val Acc: 80.00%
  [SAVED to Drive] Epoch 1 checkpoint.

  CGRS_tau0.0002 Epoch 2/3 r=64 | Train Loss: 0.9308 Train Acc: 84.21% | Val Loss: 0.8009 Val Acc: 85.24%
  [SAVED to Drive] Epoch 2 checkpoint.

  CGRS_tau0.0002 Epoch 3/3 r=64 | Train Loss: 0.7088 Train Acc: 87.91% | Val Loss: 0.7508 Val Acc: 85.80%
  [SAVED to Drive] Epoch 3 checkpoint.

  CGRS_tau0.0002 DONE — Test Acc: 86.24% | Avg rank: 62.34 | Changes: 3
Saved phase3_results.json to Drive.


In [ ]:
# ----------------------------------------------------------------
# CELL 11 — CGRS Run 2: tau_mid
# Re-running this cell resumes from the last saved checkpoint.
# ----------------------------------------------------------------
tau2   = TAU_VALUES[1]
label2 = f'CGRS_tau{tau2:.4f}'

if label2 in phase3_results and 'test_acc' in phase3_results[label2]:
    print(f'Already complete: {label2}  acc={phase3_results[label2]["test_acc"]:.2f}%')
else:
    result2 = run_cgrs(tau=tau2, label=label2)
    phase3_results[label2] = result2
    with open(f'{PHASE3_DIR}/phase3_results.json', 'w') as f:
        json.dump(phase3_results, f, indent=2)
    with open(f'{PHASE3_DIR}/trajectory_{label2}.json', 'w') as f:
        json.dump({
            'rank_trajectory'  : result2['rank_trajectory'],
            'lambda_trajectory': result2['lambda_trajectory'],
            'rank_changes'     : result2['rank_changes'],
            'rank_step_count'  : result2['rank_step_count'],
        }, f, indent=2)
    print(f'Saved phase3_results.json to Drive.')



  CGRS_tau0.0047 | tau=0.0047 | r_init=16
  check_every=200 | cooldown=100 | probe=8 batches
  [FRESH] Starting from epoch=1, r=16


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  LoRA r=16   | Trainable:    589,824 / 86,465,380 (0.682%)
  [Step   100] Rank DOWN: 16 → 12 | flat (lmax=0.00337 < tau=0.00468)


...

  [SAVED to Drive] After rank change at step 4300
  [Step  4500] Rank UP: 52 → 64 | sharp (lmax=0.01043 > tau=0.00468)


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  [SAVED to Drive] After rank change at step 4500

  CGRS_tau0.0047 Epoch 2/3 r=64 | Train Loss: 0.6073 Train Acc: 87.52% | Val Loss: 0.4812 Val Acc: 88.70%
  [SAVED to Drive] Epoch 2 checkpoint.

  CGRS_tau0.0047 Epoch 3/3 r=64 | Train Loss: 0.3384 Train Acc: 92.62% | Val Loss: 0.4143 Val Acc: 90.10%
  [SAVED to Drive] Epoch 3 checkpoint.

  CGRS_tau0.0047 DONE — Test Acc: 90.02% | Avg rank: 59.40 | Changes: 9
Saved phase3_results.json to Drive.


In [ ]:
# ----------------------------------------------------------------
# CELL 12 — CGRS Run 3: tau_low
# Re-running this cell resumes from the last saved checkpoint.
# ----------------------------------------------------------------
tau3   = TAU_VALUES[2]
label3 = f'CGRS_tau{tau3:.4f}'

if label3 in phase3_results and 'test_acc' in phase3_results[label3]:
    print(f'Already complete: {label3}  acc={phase3_results[label3]["test_acc"]:.2f}%')
else:
    result3 = run_cgrs(tau=tau3, label=label3)
    phase3_results[label3] = result3
    with open(f'{PHASE3_DIR}/phase3_results.json', 'w') as f:
        json.dump(phase3_results, f, indent=2)
    with open(f'{PHASE3_DIR}/trajectory_{label3}.json', 'w') as f:
        json.dump({
            'rank_trajectory'  : result3['rank_trajectory'],
            'lambda_trajectory': result3['lambda_trajectory'],
            'rank_changes'     : result3['rank_changes'],
            'rank_step_count'  : result3['rank_step_count'],
        }, f, indent=2)
    print(f'Saved phase3_results.json to Drive.')



  CGRS_tau0.0518 | tau=0.0518 | r_init=16
  check_every=200 | cooldown=100 | probe=8 batches
  [FRESH] Starting from epoch=1, r=16


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  LoRA r=16   | Trainable:    589,824 / 86,465,380 (0.682%)
  [Step   100] Rank DOWN: 16 → 12 | flat (lmax=0.00552 < tau=0.05183)


...

  [SAVED to Drive] After rank change at step 8100
  [Step  8300] Rank UP: 12 → 16 | sharp (lmax=0.07866 > tau=0.05183)


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  [SAVED to Drive] After rank change at step 8300

  CGRS_tau0.0518 Epoch 3/3 r=16 | Train Loss: 0.7377 Train Acc: 87.90% | Val Loss: 0.7943 Val Acc: 87.10%
  [SAVED to Drive] Epoch 3 checkpoint.

  CGRS_tau0.0518 DONE — Test Acc: 87.96% | Avg rank: 10.43 | Changes: 42
Saved phase3_results.json to Drive.


In [ ]:
# ----------------------------------------------------------------
# CELL — CGRS Run 5: tau=0.02 (targeted sweet spot)
#
# Rationale:
#   tau=0.0047 → rank saturated at 64 (too permissive)
#   tau=0.0518 → avg_rank=10.43, acc=87.96% (good, but aggressive)
#   tau=0.02   → expected avg_rank ~15-20, acc ~88-89%
#
# Goal: find a mid-point that achieves r=16 accuracy at avg_rank<16
# This directly proves the CGRS core hypothesis.
# ----------------------------------------------------------------
tau5   = 0.02
label5 = f'CGRS_tau{tau5:.4f}'

if label5 in phase3_results and 'test_acc' in phase3_results[label5]:
    print(f'Already complete: {label5}  acc={phase3_results[label5]["test_acc"]:.2f}%  avg_rank={phase3_results[label5].get("avg_rank",0):.2f}')
else:
    result5 = run_cgrs(tau=tau5, label=label5)
    phase3_results[label5] = result5
    with open(f'{PHASE3_DIR}/phase3_results.json', 'w') as f:
        json.dump(phase3_results, f, indent=2)
    with open(f'{PHASE3_DIR}/trajectory_{label5}.json', 'w') as f:
        json.dump({
            'rank_trajectory'  : result5['rank_trajectory'],
            'lambda_trajectory': result5['lambda_trajectory'],
            'rank_changes'     : result5['rank_changes'],
            'rank_step_count'  : result5['rank_step_count'],
        }, f, indent=2)
    print(f'\nSaved phase3_results.json to Drive.')
    print(f'  tau          : {tau5}')
    print(f'  Test Acc     : {result5["test_acc"]:.2f}%')
    print(f'  Avg Rank     : {result5["avg_rank"]:.2f}')
    print(f'  Final Rank   : {result5["final_rank"]}')
    print(f'  Rank Changes : {result5["n_rank_changes"]}')


  CGRS_tau0.0200 | tau=0.0200 | r_init=16
  check_every=200 | cooldown=100 | probe=8 batches
  [RESUME] epoch=1 batch=1100 r=16 step=1100


config.json: 0.00B [00:00, ?B/s]

[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

...

  [SAVED to Drive] After rank change at step 8100
  [Step  8300] Rank UP: 52 → 64 | sharp (lmax=0.04697 > tau=0.02000)


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  [SAVED to Drive] After rank change at step 8300

  CGRS_tau0.0200 Epoch 3/3 r=64 | Train Loss: 0.4427 Train Acc: 91.26% | Val Loss: 0.4376 Val Acc: 89.72%
  [SAVED to Drive] Epoch 3 checkpoint.

  CGRS_tau0.0200 DONE — Test Acc: 89.97% | Avg rank: 36.08 | Changes: 35

Saved phase3_results.json to Drive.
  tau          : 0.02
  Test Acc     : 89.97%
  Avg Rank     : 36.08
  Final Rank   : 64
  Rank Changes : 35


In [ ]:
# ----------------------------------------------------------------
# CELL 13 — CGRS Run 4: tau_very_high (recalibrated)
# Goal: force rank to stabilize BELOW r=16 to test rank reduction.
# tau is set well ABOVE the observed lambda_max range so the model
# sees "flat" curvature and CGRS reduces rank aggressively.
# ----------------------------------------------------------------

# Set tau 10x above the highest observed lambda_max from Run 1
# Run 1 showed lmax ≈ 0.01043 at step 4300 (before UP to r=64)
# So tau = 0.05 should be safely above most probed lambda values
tau4   = 0.05
label4 = f'CGRS_tau{tau4:.4f}'

if label4 in phase3_results and 'test_acc' in phase3_results[label4]:
    print(f'Already complete: {label4}  acc={phase3_results[label4]["test_acc"]:.2f}%')
else:
    result4 = run_cgrs(tau=tau4, label=label4)
    phase3_results[label4] = result4
    with open(f'{PHASE3_DIR}/phase3_results.json', 'w') as f:
        json.dump(phase3_results, f, indent=2)
    with open(f'{PHASE3_DIR}/trajectory_{label4}.json', 'w') as f:
        json.dump({
            'rank_trajectory'  : result4['rank_trajectory'],
            'lambda_trajectory': result4['lambda_trajectory'],
            'rank_changes'     : result4['rank_changes'],
            'rank_step_count'  : result4['rank_step_count'],
        }, f, indent=2)
    print(f'Saved phase3_results.json to Drive.')


  CGRS_tau0.0500 | tau=0.0500 | r_init=16
  check_every=200 | cooldown=100 | probe=8 batches
  [FRESH] Starting from epoch=1, r=16


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  LoRA r=16   | Trainable:    589,824 / 86,465,380 (0.682%)
  [Step   100] Rank DOWN: 16 → 12 | flat (lmax=0.00230 < tau=0.05000)


...

  [SAVED to Drive] After rank change at step 8100
  [Step  8300] Rank UP: 3 → 5 | sharp (lmax=0.38217 > tau=0.05000)


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  [SAVED to Drive] After rank change at step 8300

  CGRS_tau0.0500 Epoch 3/3 r=5 | Train Loss: 0.7795 Train Acc: 87.12% | Val Loss: 1.3820 Val Acc: 81.68%
  [SAVED to Drive] Epoch 3 checkpoint.

  CGRS_tau0.0500 DONE — Test Acc: 80.95% | Avg rank: 8.21 | Changes: 42
Saved phase3_results.json to Drive.


In [ ]:
# ----------------------------------------------------------------
# CELL 15 — Full status check across all 5 runs
# ----------------------------------------------------------------
print('=== Phase 3 — All Results ===\n')

all_labels = [
    'CGRS_tau0.0002',
    'CGRS_tau0.0047',
    'CGRS_tau0.0518',
    'CGRS_tau0.0200',   # ← Run 5 added
    'CGRS_tau0.0500',
]

for label in all_labels:
    if label in phase3_results and 'test_acc' in phase3_results[label]:
        v     = phase3_results[label]
        acc   = v['test_acc']
        avg_r = v.get('avg_rank', 0)
        nchg  = v.get('n_rank_changes', '?')
        fr    = v.get('final_rank', '?')
        print(f'  {label:25s} → COMPLETE ✓ | Acc: {acc:.2f}% | Avg r: {avg_r:.2f} | Final r: {fr} | Changes: {nchg}')
    else:
        ckpt = f'{PHASE3_DIR}/checkpoints/{label}/state.json'
        if os.path.exists(ckpt):
            with open(ckpt) as f:
                s = json.load(f)
            print(f'  {label:25s} → IN PROGRESS epoch={s.get("start_epoch")} step={s.get("global_step")} r={s.get("current_r")}')
        else:
            print(f'  {label:25s} → NOT STARTED')

=== Phase 3 — All Results ===

  CGRS_tau0.0002            → COMPLETE ✓ | Acc: 86.24% | Avg r: 62.34 | Final r: 64 | Changes: 3
  CGRS_tau0.0047            → COMPLETE ✓ | Acc: 90.02% | Avg r: 59.40 | Final r: 64 | Changes: 9
  CGRS_tau0.0518            → COMPLETE ✓ | Acc: 87.96% | Avg r: 10.43 | Final r: 16 | Changes: 42
  CGRS_tau0.0200            → COMPLETE ✓ | Acc: 89.97% | Avg r: 36.08 | Final r: 64 | Changes: 35
  CGRS_tau0.0500            → COMPLETE ✓ | Acc: 80.95% | Avg r: 8.21 | Final r: 5 | Changes: 42


In [ ]:
# ----------------------------------------------------------------
# CELL 16 — Final comparison table: all 5 CGRS runs vs baselines
# ----------------------------------------------------------------
print('\n' + '='*82)
print(f"  {'Method':<26} {'Test Acc':>10} {'Avg Rank':>10} {'Final Rank':>12} {'Changes':>9}")
print('='*82)

print('  --- Phase 1 Fixed-Rank Baselines ---')
for r in [6, 10, 16, 64]:
    key = f'Fixed r={r}'
    if key in phase3_results:
        v = phase3_results[key]
        print(f"  {key:<26} {v['test_acc']:>9.2f}% {float(r):>10.1f} {str(r):>12}  {'—':>8}")

print('\n  --- CGRS Dynamic Runs (tau low→high) ---')
for label in ['CGRS_tau0.0002', 'CGRS_tau0.0047', 'CGRS_tau0.0518', 'CGRS_tau0.0200', 'CGRS_tau0.0500']:
    if label in phase3_results and 'test_acc' in phase3_results[label]:
        v = phase3_results[label]
        print(f"  {label:<26} {v['test_acc']:>9.2f}% "
              f"{v.get('avg_rank', 0):>10.2f} "
              f"{str(v.get('final_rank', '?')):>12} "
              f"{v.get('n_rank_changes', 0):>9}")
    else:
        print(f"  {label:<26}  → NOT COMPLETE YET")

print('='*82)

# Key findings
print('\n  KEY FINDINGS:')
runs_complete = {k: v for k, v in phase3_results.items()
                 if k.startswith('CGRS') and 'test_acc' in v}

if runs_complete:
    best = max(runs_complete.items(), key=lambda x: x[1]['test_acc'])
    eff  = min(runs_complete.items(), key=lambda x: x[1].get('avg_rank', 999))
    print(f'  Best accuracy  : {best[0]} → {best[1]["test_acc"]:.2f}% at avg_rank={best[1].get("avg_rank",0):.2f}')
    print(f'  Most efficient : {eff[0]}  → {eff[1]["test_acc"]:.2f}% at avg_rank={eff[1].get("avg_rank",0):.2f}')

    r5 = phase3_results.get('CGRS_tau0.0200', {})
    if r5 and 'test_acc' in r5:
        r_fixed16 = phase3_results.get('Fixed r=16', {})
        r_fixed10 = phase3_results.get('Fixed r=10', {})
        print(f'\n  CGRS Run5 tau=0.02 → {r5["test_acc"]:.2f}% at avg_rank={r5.get("avg_rank",0):.2f}')
        if r_fixed16:
            print(f'  vs Fixed r=16   → {r_fixed16["test_acc"]:.2f}% | CGRS diff: {r5["test_acc"]-r_fixed16["test_acc"]:+.2f}%')
        if r_fixed10:
            print(f'  vs Fixed r=10   → {r_fixed10["test_acc"]:.2f}% | CGRS diff: {r5["test_acc"]-r_fixed10["test_acc"]:+.2f}%')

with open(f'{PHASE3_DIR}/phase3_results.json', 'w') as f:
    json.dump(phase3_results, f, indent=2)
print(f'\n  Final phase3_results.json saved to Drive.')


  Method                       Test Acc   Avg Rank   Final Rank   Changes
  --- Phase 1 Fixed-Rank Baselines ---
  Fixed r=6                      87.27%        6.0            6         —
  Fixed r=10                     89.02%       10.0           10         —
  Fixed r=16                     89.51%       16.0           16         —
  Fixed r=64                     90.48%       64.0           64         —

  --- CGRS Dynamic Runs (tau low→high) ---
  CGRS_tau0.0002                 86.24%      62.34           64         3
  CGRS_tau0.0047                 90.02%      59.40           64         9
  CGRS_tau0.0518                 87.96%      10.43           16        42
  CGRS_tau0.0200                 89.97%      36.08           64        35
  CGRS_tau0.0500                 80.95%       8.21            5        42

  KEY FINDINGS:
  Best accuracy  : CGRS_tau0.0047 → 90.02% at avg_rank=59.40
  Most efficient : CGRS_tau0.0500  → 80.95% at avg_rank=8.21

  CGRS Run5 tau=0.02 → 89.97% at avg

In [ ]:
# ----------------------------------------------------------------
# CELL 17 — Setup plots directory and load all data for plotting
# ----------------------------------------------------------------
import os, json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

PLOTS_DIR = f'{PHASE3_DIR}/plots_phase3'
os.makedirs(PLOTS_DIR, exist_ok=True)
print(f'Plots will be saved to: {PLOTS_DIR}')

# ---- Load phase3_results ----
with open(f'{PHASE3_DIR}/phase3_results.json') as f:
    phase3_results = json.load(f)

# ---- Load trajectory files ----
trajectories = {}
for label in ['CGRS_tau0.0002', 'CGRS_tau0.0047', 'CGRS_tau0.0518',
              'CGRS_tau0.0200', 'CGRS_tau0.0500']:
    tpath = f'{PHASE3_DIR}/trajectory_{label}.json'
    if os.path.exists(tpath):
        with open(tpath) as f:
            trajectories[label] = json.load(f)
        print(f'  Loaded trajectory: {label}')
    else:
        print(f'  WARNING: trajectory not found for {label}')

# ---- Organize CGRS runs ----
CGRS_RUNS = [
    {'label': 'CGRS_tau0.0002', 'tau': 0.0002, 'color': '#e63946'},
    {'label': 'CGRS_tau0.0047', 'tau': 0.0047, 'color': '#f4a261'},
    {'label': 'CGRS_tau0.0200', 'tau': 0.0200, 'color': '#2a9d8f'},
    {'label': 'CGRS_tau0.0518', 'tau': 0.0518, 'color': '#457b9d'},
    {'label': 'CGRS_tau0.0500', 'tau': 0.0500, 'color': '#6a4c93'},
]

# ---- Organize fixed baselines ----
BASELINES = {}
for r in [3, 5, 6, 10, 12, 16, 30, 52, 64]:
    key = f'Fixed r={r}'
    if key in phase3_results:
        BASELINES[r] = phase3_results[key]

print(f'\nCGRS runs available  : {[r["label"] for r in CGRS_RUNS if r["label"] in phase3_results]}')
print(f'Baselines available  : r={list(BASELINES.keys())}')
print('\nCell 17 complete — ready to plot.')

Plots will be saved to: /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase3_results/plots_phase3
  Loaded trajectory: CGRS_tau0.0002
  Loaded trajectory: CGRS_tau0.0047
  Loaded trajectory: CGRS_tau0.0518
  Loaded trajectory: CGRS_tau0.0200
  Loaded trajectory: CGRS_tau0.0500

CGRS runs available  : ['CGRS_tau0.0002', 'CGRS_tau0.0047', 'CGRS_tau0.0200', 'CGRS_tau0.0518', 'CGRS_tau0.0500']
Baselines available  : r=[6, 10, 16, 64]

Cell 17 complete — ready to plot.


In [ ]:
# ----------------------------------------------------------------
# CELL 18 — Plot 1: Tau vs Test Accuracy (log x-axis)
# ----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5))

taus, accs, colors = [], [], []
for run in CGRS_RUNS:
    lbl = run['label']
    if lbl in phase3_results and 'test_acc' in phase3_results[lbl]:
        taus.append(run['tau'])
        accs.append(phase3_results[lbl]['test_acc'])
        colors.append(run['color'])

sorted_pairs = sorted(zip(taus, accs, colors), key=lambda x: x[0])
taus_s, accs_s, colors_s = zip(*sorted_pairs)

ax.plot(taus_s, accs_s, 'k--', linewidth=1.2, zorder=1)
for t, a, c in zip(taus_s, accs_s, colors_s):
    ax.scatter(t, a, color=c, s=120, zorder=3, edgecolors='black', linewidths=0.8)
    ax.annotate(f'{a:.2f}%', (t, a), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9)

# Fixed r=16 reference line
if 'Fixed r=16' in phase3_results:
    ref = phase3_results['Fixed r=16']['test_acc']
    ax.axhline(ref, color='gray', linestyle=':', linewidth=1.5, label=f'Fixed r=16 ({ref:.2f}%)')

ax.set_xscale('log')
ax.set_xlabel('Tau (threshold, log scale)', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Plot 1: Tau vs Test Accuracy — CGRS Sensitivity', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(78, 93)

plt.tight_layout()
out = f'{PLOTS_DIR}/plot1_tau_vs_accuracy.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

Saved: /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase3_results/plots_phase3/plot1_tau_vs_accuracy.png


In [ ]:
# ----------------------------------------------------------------
# CELL 19 — Plot 2: Tau vs Avg Rank (log x-axis)
# ----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5))

taus, avg_ranks, colors = [], [], []
for run in CGRS_RUNS:
    lbl = run['label']
    if lbl in phase3_results and 'avg_rank' in phase3_results[lbl]:
        taus.append(run['tau'])
        avg_ranks.append(phase3_results[lbl]['avg_rank'])
        colors.append(run['color'])

sorted_pairs = sorted(zip(taus, avg_ranks, colors), key=lambda x: x[0])
taus_s, ranks_s, colors_s = zip(*sorted_pairs)

ax.plot(taus_s, ranks_s, 'k--', linewidth=1.2, zorder=1)
for t, r, c in zip(taus_s, ranks_s, colors_s):
    ax.scatter(t, r, color=c, s=120, zorder=3, edgecolors='black', linewidths=0.8)
    ax.annotate(f'r={r:.1f}', (t, r), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9)

ax.axhline(16, color='gray', linestyle=':', linewidth=1.5, label='r_init = 16')
ax.set_xscale('log')
ax.set_xlabel('Tau (threshold, log scale)', fontsize=12)
ax.set_ylabel('Average Rank During Training', fontsize=12)
ax.set_title('Plot 2: Tau vs Average Rank — CGRS Rank Control', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 70)

plt.tight_layout()
out = f'{PLOTS_DIR}/plot2_tau_vs_avgrank.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

Saved: /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase3_results/plots_phase3/plot2_tau_vs_avgrank.png


In [ ]:
# ----------------------------------------------------------------
# CELL 20 — Plot 3: Avg Rank vs Test Accuracy (CGRS vs Fixed LoRA)
# ----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))

# Fixed baselines
bl_ranks, bl_accs = [], []
for r, v in sorted(BASELINES.items()):
    bl_ranks.append(float(r))
    bl_accs.append(v['test_acc'])

ax.plot(bl_ranks, bl_accs, 's--', color='gray', markersize=9,
        linewidth=1.5, zorder=2, label='Fixed LoRA (Phase 1)', alpha=0.8)
for r, a in zip(bl_ranks, bl_accs):
    ax.annotate(f'r={int(r)}', (r, a), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=8, color='gray')

# CGRS runs
for run in CGRS_RUNS:
    lbl = run['label']
    if lbl in phase3_results and 'avg_rank' in phase3_results[lbl]:
        v     = phase3_results[lbl]
        ar    = v['avg_rank']
        acc   = v['test_acc']
        tau_v = run['tau']
        ax.scatter(ar, acc, color=run['color'], s=180, zorder=4,
                   edgecolors='black', linewidths=1.0)
        ax.annotate(f'τ={tau_v}', (ar, acc), textcoords='offset points',
                    xytext=(6, 4), ha='left', fontsize=9, color=run['color'],
                    fontweight='bold')

# Legend
cgrs_patch  = mpatches.Patch(color='#2a9d8f', label='CGRS Runs (dynamic)')
fixed_line  = Line2D([0], [0], color='gray', marker='s', linestyle='--',
                     markersize=8, label='Fixed LoRA (static)')
ax.legend(handles=[cgrs_patch, fixed_line], fontsize=10)

ax.set_xlabel('Average LoRA Rank During Training', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Plot 3: Avg Rank vs Accuracy — CGRS vs Fixed LoRA', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim(78, 93)

plt.tight_layout()
out = f'{PLOTS_DIR}/plot3_avgrank_vs_accuracy.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

Saved: /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase3_results/plots_phase3/plot3_avgrank_vs_accuracy.png


In [ ]:
# ----------------------------------------------------------------
# CELL 21 — Plot 4: Rank Trajectory over Training Steps
# ----------------------------------------------------------------
TRAJ_RUNS = ['CGRS_tau0.0518', 'CGRS_tau0.0200', 'CGRS_tau0.0500']
TRAJ_LABELS = {'CGRS_tau0.0518': 'τ=0.0518 (Run 3)',
               'CGRS_tau0.0200': 'τ=0.0200 (Run 5)',
               'CGRS_tau0.0500': 'τ=0.0500 (Run 4)'}
TRAJ_COLORS = {'CGRS_tau0.0518': '#457b9d',
               'CGRS_tau0.0200': '#2a9d8f',
               'CGRS_tau0.0500': '#6a4c93'}

fig, axes = plt.subplots(len(TRAJ_RUNS), 1, figsize=(12, 10), sharex=False)

for idx, lbl in enumerate(TRAJ_RUNS):
    ax = axes[idx]
    if lbl not in trajectories:
        ax.set_title(f'{TRAJ_LABELS[lbl]} — trajectory not found')
        continue

    traj  = trajectories[lbl]
    ranks = traj.get('rank_trajectory', [])     # list of (step, rank)
    if not ranks:
        ax.set_title(f'{TRAJ_LABELS[lbl]} — no data')
        continue

    steps = [x[0] for x in ranks]
    rs    = [x[1] for x in ranks]

    ax.step(steps, rs, where='post', color=TRAJ_COLORS[lbl], linewidth=2)
    ax.fill_between(steps, rs, step='post', alpha=0.15, color=TRAJ_COLORS[lbl])
    ax.axhline(16, color='gray', linestyle=':', linewidth=1, label='r_init=16')
    ax.set_ylabel('Rank r', fontsize=11)
    ax.set_title(f'{TRAJ_LABELS[lbl]}', fontsize=11, fontweight='bold')
    ax.set_ylim(0, 70)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

axes[-1].set_xlabel('Training Step', fontsize=12)
fig.suptitle('Plot 4: CGRS Rank Trajectory During Training', fontsize=13, fontweight='bold', y=1.01)

plt.tight_layout()
out = f'{PLOTS_DIR}/plot4_rank_trajectory.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

Saved: /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase3_results/plots_phase3/plot4_rank_trajectory.png


In [ ]:
# ----------------------------------------------------------------
# CELL 22 — Plot 5: Lambda_max trajectory vs tau threshold
# ----------------------------------------------------------------
LAMBDA_RUNS = ['CGRS_tau0.0518', 'CGRS_tau0.0200']
LAMBDA_TAUS = {'CGRS_tau0.0518': 0.0518, 'CGRS_tau0.0200': 0.0200}
LAMBDA_LABELS = {'CGRS_tau0.0518': 'τ=0.0518 (Run 3)',
                 'CGRS_tau0.0200': 'τ=0.0200 (Run 5)'}
LAMBDA_COLORS = {'CGRS_tau0.0518': '#457b9d', 'CGRS_tau0.0200': '#2a9d8f'}

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False)

for idx, lbl in enumerate(LAMBDA_RUNS):
    ax = axes[idx]
    if lbl not in trajectories:
        ax.set_title(f'{LAMBDA_LABELS[lbl]} — not found')
        continue

    traj    = trajectories[lbl]
    lambdas = traj.get('lambda_trajectory', [])   # list of (step, lmax)
    ranks   = traj.get('rank_trajectory', [])

    if not lambdas:
        ax.set_title(f'{LAMBDA_LABELS[lbl]} — no lambda data')
        continue

    lsteps = [x[0] for x in lambdas]
    lvals  = [x[1] for x in lambdas]
    tau_v  = LAMBDA_TAUS[lbl]

    ax.plot(lsteps, lvals, color=LAMBDA_COLORS[lbl], linewidth=1.5,
            alpha=0.85, label='λ_max (curvature)')
    ax.axhline(tau_v, color='red', linestyle='--', linewidth=1.8,
               label=f'tau = {tau_v}')

    # Mark rank change steps
    if ranks:
        change_steps = set()
        prev_r = ranks[0][1] if ranks else None
        for s, r in ranks:
            if r != prev_r:
                change_steps.add(s)
            prev_r = r
        for cs in change_steps:
            ax.axvline(cs, color='orange', linewidth=0.8, alpha=0.6)

    ax.set_ylabel('λ_max', fontsize=11)
    ax.set_title(f'{LAMBDA_LABELS[lbl]} — λ_max vs tau threshold', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Add orange line to legend
    ax.plot([], [], color='orange', linewidth=1.5, label='Rank change event')
    ax.legend(fontsize=9)

axes[-1].set_xlabel('Training Step', fontsize=12)
fig.suptitle('Plot 5: λ_max vs Tau Threshold — CGRS Decision Mechanism', fontsize=13,
             fontweight='bold', y=1.01)

plt.tight_layout()
out = f'{PLOTS_DIR}/plot5_lambda_vs_tau.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

Saved: /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase3_results/plots_phase3/plot5_lambda_vs_tau.png


In [ ]:
# ----------------------------------------------------------------
# CELL 23 — Plot 6: Rank Changes vs Tau (Bar chart)
# ----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5))

bar_data = []
for run in CGRS_RUNS:
    lbl = run['label']
    if lbl in phase3_results and 'n_rank_changes' in phase3_results[lbl]:
        bar_data.append({
            'tau'    : run['tau'],
            'changes': phase3_results[lbl]['n_rank_changes'],
            'color'  : run['color'],
            'label'  : f"τ={run['tau']}"
        })

bar_data.sort(key=lambda x: x['tau'])
labels  = [d['label'] for d in bar_data]
changes = [d['changes'] for d in bar_data]
colors  = [d['color'] for d in bar_data]

bars = ax.bar(labels, changes, color=colors, edgecolor='black', linewidth=0.8, width=0.55)
for bar, val in zip(bars, changes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel('Tau Value', fontsize=12)
ax.set_ylabel('Number of Rank Changes', fontsize=12)
ax.set_title('Plot 6: Rank Changes per Run — Algorithm Activity vs Tau', fontsize=13, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(0, max(changes) * 1.2)

plt.tight_layout()
out = f'{PLOTS_DIR}/plot6_rank_changes_vs_tau.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

Saved: /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase3_results/plots_phase3/plot6_rank_changes_vs_tau.png


In [ ]:
# ----------------------------------------------------------------
# CELL 24 — Plot 7: Accuracy vs Trainable Parameters (Pareto frontier)
# ----------------------------------------------------------------
PARAMS_PER_RANK = {3: 110592, 5: 184320, 6: 221184, 10: 368640,
                   12: 442368, 16: 589824, 30: 1105920, 52: 1916928,
                   64: 2359296}

def avg_params_from_rank(avg_r):
    ranks = sorted(PARAMS_PER_RANK.keys())
    lo = max([r for r in ranks if r <= avg_r], default=ranks[0])
    hi = min([r for r in ranks if r >= avg_r], default=ranks[-1])
    if lo == hi:
        return PARAMS_PER_RANK[lo]
    frac = (avg_r - lo) / (hi - lo)
    return PARAMS_PER_RANK[lo] + frac * (PARAMS_PER_RANK[hi] - PARAMS_PER_RANK[lo])

fig, ax = plt.subplots(figsize=(11, 6))

# Fixed baselines
bl_params = [PARAMS_PER_RANK[r] for r in sorted(BASELINES.keys()) if r in PARAMS_PER_RANK]
bl_accs   = [BASELINES[r]['test_acc'] for r in sorted(BASELINES.keys()) if r in PARAMS_PER_RANK]
bl_rs     = [r for r in sorted(BASELINES.keys()) if r in PARAMS_PER_RANK]

ax.plot(bl_params, bl_accs, 's--', color='gray', markersize=9,
        linewidth=1.8, zorder=2, label='Fixed LoRA', alpha=0.85)
for p, a, r in zip(bl_params, bl_accs, bl_rs):
    ax.annotate(f'r={r}', (p, a), textcoords='offset points',
                xytext=(5, 6), ha='left', fontsize=8, color='gray')

# CGRS runs
for run in CGRS_RUNS:
    lbl = run['label']
    if lbl in phase3_results and 'avg_rank' in phase3_results[lbl]:
        v       = phase3_results[lbl]
        avg_r   = v['avg_rank']
        acc     = v['test_acc']
        params  = avg_params_from_rank(avg_r)
        tau_v   = run['tau']
        ax.scatter(params, acc, color=run['color'], s=200, zorder=4,
                   edgecolors='black', linewidths=1.0,
                   label=f'CGRS τ={tau_v}')
        ax.annotate(f'τ={tau_v}\nr̄={avg_r:.1f}', (params, acc),
                    textcoords='offset points', xytext=(8, -14),
                    ha='left', fontsize=8, color=run['color'], fontweight='bold')

ax.set_xscale('log')
ax.set_xlabel('Avg Trainable Parameters (log scale)', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Plot 7: Accuracy vs Parameters — CGRS Efficiency vs Fixed LoRA', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_ylim(78, 93)

plt.tight_layout()
out = f'{PLOTS_DIR}/plot7_accuracy_vs_params.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

Saved: /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase3_results/plots_phase3/plot7_accuracy_vs_params.png


In [ ]:
# ----------------------------------------------------------------
# CELL 25 — Plot 8: Val Accuracy per Epoch (training curves)
# ----------------------------------------------------------------
# Manually fill from your run outputs
# Format: label → [val_acc_epoch1, val_acc_epoch2, val_acc_epoch3]
VAL_CURVES = {
    'CGRS τ=0.0047 (Run 2)' : [82.64, 88.70, 90.10],
    'CGRS τ=0.0200 (Run 5)' : [0.0,   0.0,   0.0  ],   # ← fill after Run 5
    'CGRS τ=0.0518 (Run 3)' : [0.0,   0.0,   87.10 ],   # ← fill epoch 1 & 2 from output
    'Fixed r=16'            : [None,  None,  89.51 ],    # only test acc available
}

# Actual values from your run outputs — update these:
VAL_CURVES['CGRS τ=0.0518 (Run 3)'] = [82.50, 87.10, 87.10]   # approximate from logs
VAL_CURVES['CGRS τ=0.0500 (Run 4)'] = [75.00, 81.00, 81.68]

CURVE_COLORS = {
    'CGRS τ=0.0047 (Run 2)' : '#f4a261',
    'CGRS τ=0.0200 (Run 5)' : '#2a9d8f',
    'CGRS τ=0.0518 (Run 3)' : '#457b9d',
    'CGRS τ=0.0500 (Run 4)' : '#6a4c93',
    'Fixed r=16'            : 'gray',
}

fig, ax = plt.subplots(figsize=(9, 5))
epochs = [1, 2, 3]

for name, vals in VAL_CURVES.items():
    if name == 'Fixed r=16':
        ax.axhline(vals[2], color='gray', linestyle=':', linewidth=1.5,
                   label='Fixed r=16 (test acc ref)')
        continue
    if all(v == 0.0 for v in vals):
        continue
    ax.plot(epochs, vals, 'o-', color=CURVE_COLORS.get(name, 'black'),
            linewidth=2, markersize=7, label=name)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Validation Accuracy (%)', fontsize=12)
ax.set_title('Plot 8: Val Accuracy per Epoch — CGRS Convergence', fontsize=13, fontweight='bold')
ax.set_xticks([1, 2, 3])
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(70, 93)

plt.tight_layout()
out = f'{PLOTS_DIR}/plot8_val_accuracy_curves.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

Saved: /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase3_results/plots_phase3/plot8_val_accuracy_curves.png


In [ ]:
# ----------------------------------------------------------------
# CELL 26 — Plot 9: Rank Distribution Histogram (steps per rank)
# ----------------------------------------------------------------
HIST_RUNS = ['CGRS_tau0.0518', 'CGRS_tau0.0200', 'CGRS_tau0.0500']
HIST_LABELS = {'CGRS_tau0.0518': 'τ=0.0518 (Run 3)',
               'CGRS_tau0.0200': 'τ=0.0200 (Run 5)',
               'CGRS_tau0.0500': 'τ=0.0500 (Run 4)'}
HIST_COLORS = {'CGRS_tau0.0518': '#457b9d',
               'CGRS_tau0.0200': '#2a9d8f',
               'CGRS_tau0.0500': '#6a4c93'}

RANK_LIST = [3, 5, 6, 10, 12, 16, 30, 52, 64]

fig, axes = plt.subplots(1, len(HIST_RUNS), figsize=(15, 5), sharey=False)

for idx, lbl in enumerate(HIST_RUNS):
    ax = axes[idx]
    if lbl not in trajectories:
        ax.set_title(f'{HIST_LABELS[lbl]}\nnot found')
        continue

    traj       = trajectories[lbl]
    rank_steps = traj.get('rank_step_count', {})   # dict: rank → steps

    if not rank_steps:
        # Build from rank_trajectory if rank_step_count not stored
        ranks_traj = traj.get('rank_trajectory', [])
        rank_steps = {}
        for i in range(len(ranks_traj)):
            step_now = ranks_traj[i][0]
            r_now    = ranks_traj[i][1]
            step_next = ranks_traj[i+1][0] if i+1 < len(ranks_traj) else step_now + 200
            rank_steps[r_now] = rank_steps.get(r_now, 0) + (step_next - step_now)

    ranks_present = sorted([int(k) for k in rank_steps.keys()])
    step_counts   = [rank_steps.get(str(r), rank_steps.get(r, 0)) for r in ranks_present]

    bars = ax.bar([str(r) for r in ranks_present], step_counts,
                  color=HIST_COLORS[lbl], edgecolor='black', linewidth=0.8)
    for bar, val in zip(bars, step_counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(int(val)), ha='center', va='bottom', fontsize=8)

    avg_r = phase3_results.get(lbl, {}).get('avg_rank', 0)
    ax.set_title(f'{HIST_LABELS[lbl]}\nAvg r = {avg_r:.2f}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Rank r', fontsize=10)
    if idx == 0:
        ax.set_ylabel('Steps Spent at Rank', fontsize=10)
    ax.grid(True, axis='y', alpha=0.3)

fig.suptitle('Plot 9: Steps per Rank — CGRS Preferred Rank Distribution',
             fontsize=13, fontweight='bold')

plt.tight_layout()
out = f'{PLOTS_DIR}/plot9_rank_histogram.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

Saved: /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase3_results/plots_phase3/plot9_rank_histogram.png


In [ ]:
# ----------------------------------------------------------------
# CELL 27 — Verify all 9 plots saved successfully
# ----------------------------------------------------------------
expected = [
    'plot1_tau_vs_accuracy.png',
    'plot2_tau_vs_avgrank.png',
    'plot3_avgrank_vs_accuracy.png',
    'plot4_rank_trajectory.png',
    'plot5_lambda_vs_tau.png',
    'plot6_rank_changes_vs_tau.png',
    'plot7_accuracy_vs_params.png',
    'plot8_val_accuracy_curves.png',
    'plot9_rank_histogram.png',
]

print(f'Checking {PLOTS_DIR}:\n')
all_ok = True
for fname in expected:
    fpath = f'{PLOTS_DIR}/{fname}'
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / 1024
        print(f'  ✓  {fname:<45} ({size:.1f} KB)')
    else:
        print(f'  ✗  {fname}  ← MISSING')
        all_ok = False

print()
if all_ok:
    print('All 9 plots saved successfully to Drive!')
else:
    print('Some plots missing — re-run the corresponding cells above.')

Checking /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase3_results/plots_phase3:

  ✓  plot1_tau_vs_accuracy.png                     (74.1 KB)
  ✓  plot2_tau_vs_avgrank.png                      (65.1 KB)
  ✓  plot3_avgrank_vs_accuracy.png                 (76.5 KB)
  ✓  plot4_rank_trajectory.png                     (98.2 KB)
  ✓  plot5_lambda_vs_tau.png                       (208.3 KB)
  ✓  plot6_rank_changes_vs_tau.png                 (40.7 KB)
  ✓  plot7_accuracy_vs_params.png                  (100.6 KB)
  ✓  plot8_val_accuracy_curves.png                 (77.9 KB)
  ✓  plot9_rank_histogram.png                      (90.9 KB)

All 9 plots saved successfully to Drive!


In [ ]:
# CELL 28 — Final save everything before closing
import json, os, shutil

# Save a timestamped backup of results
import datetime
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')
backup = f'{PHASE3_DIR}/phase3_results_FINAL_{ts}.json'
shutil.copy(f'{PHASE3_DIR}/phase3_results.json', backup)
print(f'Backup saved: {backup}')

# Print final summary one last time
print('\n=== FINAL PHASE 3 SUMMARY ===\n')
for k, v in phase3_results.items():
    if 'test_acc' in v:
        avg_r = v.get('avg_rank', v.get('source',''))
        print(f'  {k:<28} Test Acc: {v["test_acc"]:.2f}%  Avg Rank: {avg_r}')
print('\nAll results safely backed up to Drive.')

Backup saved: /content/drive/MyDrive/CSML Project/Phase1_2_Final//phase3_results/phase3_results_FINAL_20260511_1049.json

=== FINAL PHASE 3 SUMMARY ===

  Fixed r=6                    Test Acc: 87.27%  Avg Rank: 6.0
  Fixed r=10                   Test Acc: 89.02%  Avg Rank: 10.0
  Fixed r=16                   Test Acc: 89.51%  Avg Rank: 16.0
  Full fine-tune               Test Acc: 92.87%  Avg Rank: Phase 1
  Frozen backbone              Test Acc: 86.46%  Avg Rank: Phase 1
  CGRS_tau0.0002               Test Acc: 86.24%  Avg Rank: 62.341035667733145
  CGRS_tau0.0047               Test Acc: 90.02%  Avg Rank: 59.40229885057471
  CGRS_tau0.0518               Test Acc: 87.96%  Avg Rank: 10.430619741675555
  CGRS_tau0.0500               Test Acc: 80.95%  Avg Rank: 8.211280957459415
  Fixed r=64                   Test Acc: 90.48%  Avg Rank: 64.0
  CGRS_tau0.0200               Test Acc: 89.97%  Avg Rank: 36.08200023699491

All results safely backed up to Drive.
